In [ ]:
import torch
import torch.nn as nn
from torch.optim import RAdam
from sklearn.preprocessing import StandardScaler
import numpy as np
from scipy.integrate import solve_ivp
from sklearn.metrics import mean_absolute_error

# Utility function to compute LCM
def compute_lcm(a, b):
    a, b = int(a), int(b)  # Ensure inputs are integers
    return abs(a * b) // np.gcd(a, b)

# Define EnhancedChaoticLayer class
class EnhancedChaoticLayer(nn.Module):
    def __init__(self, iterations=500, dt=0.01, system='lorenz'):
        super(EnhancedChaoticLayer, self).__init__()
        self.iterations = iterations
        self.dt = dt
        self.system = system

    def _lorenz(self, t, state, sigma=10, beta=8/3, rho=28):
        x, y, z = state
        dx = sigma * (y - x)
        dy = x * (rho - z) - y
        dz = x * y - beta * z
        return [dx, dy, dz]

    def _rossler(self, t, state, a=0.2, b=0.2, c=5.7):
        x, y, z = state
        dx = -y - z
        dy = x + a * y
        dz = b + z * (x - c)
        return [dx, dy, dz]

    def _chen(self, t, state, a=35, b=3, c=28):
        x, y, z = state
        dx = a * (y - x)
        dy = (c - a) * x - x * z + c * y
        dz = x * y - b * z
        return [dx, dy, dz]

    def _extract_features(self, trajectory):
        x_vals, y_vals, z_vals = trajectory
        means = [np.mean(vals) for vals in [x_vals, y_vals, z_vals]]
        stds = [np.std(vals) for vals in [x_vals, y_vals, z_vals]]
        radius = np.sqrt(x_vals**2 + y_vals**2 + z_vals**2)
        divergence_rate = np.mean(np.abs(np.diff(radius)))

        features = [
            *means, *stds,
            np.mean(radius), np.std(radius),
            divergence_rate
        ]
        return features

    def forward(self, x):
        system_func = {
            'lorenz': self._lorenz,
            'rossler': self._rossler,
            'chen': self._chen
        }[self.system]

        results = []
        for sample in x:
            initial_state = [sample[0].item(), sample[1].item(), np.sqrt(sample[0].item()**2 + sample[1].item()**2)]
            sol = solve_ivp(
                system_func,
                [0, self.iterations * self.dt],
                initial_state,
                method='RK45',
                t_eval=np.linspace(0, self.iterations * self.dt, self.iterations)
            )
            features = self._extract_features(sol.y)
            results.append(features)

        return torch.tensor(results, dtype=torch.float32)

# Define EnhancedLCMNet class
class EnhancedLCMNet(nn.Module):
    def __init__(self):
        super(EnhancedLCMNet, self).__init__()
        chaotic_features = 9
        self.fc1 = nn.Linear(2, 32)
        self.bn1 = nn.BatchNorm1d(32)

        self.chaotic_lorenz = EnhancedChaoticLayer(system='lorenz')
        self.chaotic_rossler = EnhancedChaoticLayer(system='rossler')
        self.chaotic_chen = EnhancedChaoticLayer(system='chen')

        combined_features = 32 + (chaotic_features * 3)
        self.fc2 = nn.Linear(combined_features, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.fc3 = nn.Linear(64, 32)
        self.bn3 = nn.BatchNorm1d(32)
        self.fc4 = nn.Linear(32, 1)

        self.dropout = nn.Dropout(0.1)
        self.leaky_relu = nn.LeakyReLU()

    def forward(self, x):
        x_hidden = self.leaky_relu(self.bn1(self.fc1(x)))

        lorenz_features = self.chaotic_lorenz(x)
        rossler_features = self.chaotic_rossler(x)
        chen_features = self.chaotic_chen(x)

        combined = torch.cat([x_hidden, lorenz_features, rossler_features, chen_features], dim=1)

        x_out = self.leaky_relu(self.bn2(self.fc2(combined)))
        x_out = self.dropout(x_out)
        x_out = self.leaky_relu(self.bn3(self.fc3(x_out)))
        x_out = self.fc4(x_out)

        return torch.abs(x_out)

# Training function
def train_model(model, train_inputs, train_targets, val_inputs, val_targets, epochs=1000):
    criterion = nn.MSELoss()
    optimizer = RAdam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(train_inputs)
        loss = criterion(outputs, train_targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(val_inputs)
            val_loss = criterion(val_outputs, val_targets)

        scheduler.step(val_loss)

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

    return model

# Main script
if __name__ == "__main__":
    num_samples = 300
    inputs = torch.randint(1, 50, (num_samples, 2)).float()
    lcm_targets = torch.tensor([compute_lcm(a, b) for a, b in inputs.numpy()], dtype=torch.float32).unsqueeze(1)

    input_scaler = StandardScaler()
    inputs = torch.tensor(input_scaler.fit_transform(inputs), dtype=torch.float32)

    train_size = int(0.7 * num_samples)
    val_size = int(0.15 * num_samples)
    train_inputs, val_inputs, test_inputs = inputs[:train_size], inputs[train_size:train_size+val_size], inputs[train_size+val_size:]
    train_targets, val_targets, test_targets = lcm_targets[:train_size], lcm_targets[train_size:train_size+val_size], lcm_targets[train_size+val_size:]

    model = EnhancedLCMNet()
    model = train_model(model, train_inputs, train_targets, val_inputs, val_targets)

    model.eval()
    with torch.no_grad():
        test_outputs = model(test_inputs)
        test_loss = nn.MSELoss()(test_outputs, test_targets)
        print(f"\nTest Loss: {test_loss.item():.4f}")

        test_outputs_np = test_outputs.numpy()
        test_targets_np = test_targets.numpy()

        print("\nSample Predictions:")
        for i in range(10):
            orig_inputs = input_scaler.inverse_transform(test_inputs[i].numpy().reshape(1, -1))[0]
            print(f"Numbers: ({int(orig_inputs[0])}, {int(orig_inputs[1])}), Predicted LCM: {test_outputs_np[i][0]:.1f}, True LCM: {test_targets_np[i][0]}")

        mape = np.mean(np.abs((test_targets_np - test_outputs_np) / test_targets_np)) * 100
        print(f"\nMean Absolute Percentage Error: {mape:.2f}%")

Epoch 0, Train Loss: 440711.5938, Val Loss: 291299.7500
Epoch 50, Train Loss: 440305.5938, Val Loss: 290736.9375
Epoch 100, Train Loss: 439497.8438, Val Loss: 290269.6562
Epoch 150, Train Loss: 438738.7812, Val Loss: 289826.1250
Epoch 200, Train Loss: 438173.0625, Val Loss: 289457.2500
Epoch 250, Train Loss: 437583.7812, Val Loss: 289094.3438
Epoch 300, Train Loss: 436935.8125, Val Loss: 288625.1875
Epoch 350, Train Loss: 436229.0938, Val Loss: 288172.7500
Epoch 400, Train Loss: 435442.3125, Val Loss: 287623.3750
Epoch 450, Train Loss: 434465.9062, Val Loss: 287187.6562
Epoch 500, Train Loss: 434076.6875, Val Loss: 286946.6250
Epoch 550, Train Loss: 433851.0625, Val Loss: 286826.9375
Epoch 600, Train Loss: 433656.0000, Val Loss: 286691.4375
Epoch 650, Train Loss: 433582.0625, Val Loss: 286614.7188
Epoch 700, Train Loss: 433620.6875, Val Loss: 286590.0312
Epoch 750, Train Loss: 433630.7500, Val Loss: 286562.7188
Epoch 800, Train Loss: 433576.4688, Val Loss: 286582.5625
Epoch 850, Train 

In [ ]:
import torch
import torch.nn as nn
from torch.optim import RAdam
from sklearn.preprocessing import StandardScaler
import numpy as np
from scipy.integrate import solve_ivp
from sklearn.metrics import mean_absolute_error

# Utility function to compute LCM
def compute_lcm(a, b):
    a, b = int(a), int(b)  # Ensure inputs are integers
    return abs(a * b) // np.gcd(a, b)

# Define EnhancedChaoticLayer class
class EnhancedChaoticLayer(nn.Module):
    def __init__(self, iterations=500, dt=0.01, system='lorenz'):
        super(EnhancedChaoticLayer, self).__init__()
        self.iterations = iterations
        self.dt = dt
        self.system = system

    def _lorenz(self, t, state, sigma=10, beta=8/3, rho=28):
        x, y, z = state
        dx = sigma * (y - x)
        dy = x * (rho - z) - y
        dz = x * y - beta * z
        return [dx, dy, dz]

    def _rossler(self, t, state, a=0.2, b=0.2, c=5.7):
        x, y, z = state
        dx = -y - z
        dy = x + a * y
        dz = b + z * (x - c)
        return [dx, dy, dz]

    def _chen(self, t, state, a=35, b=3, c=28):
        x, y, z = state
        dx = a * (y - x)
        dy = (c - a) * x - x * z + c * y
        dz = x * y - b * z
        return [dx, dy, dz]

    def _extract_features(self, trajectory):
        x_vals, y_vals, z_vals = trajectory
        means = [np.mean(vals) for vals in [x_vals, y_vals, z_vals]]
        stds = [np.std(vals) for vals in [x_vals, y_vals, z_vals]]
        radius = np.sqrt(x_vals**2 + y_vals**2 + z_vals**2)
        divergence_rate = np.mean(np.abs(np.diff(radius)))

        features = [
            *means, *stds,
            np.mean(radius), np.std(radius),
            divergence_rate
        ]
        return features

    def forward(self, x):
        system_func = {
            'lorenz': self._lorenz,
            'rossler': self._rossler,
            'chen': self._chen
        }[self.system]

        results = []
        for sample in x:
            initial_state = [sample[0].item(), sample[1].item(), np.sqrt(sample[0].item()**2 + sample[1].item()**2)]
            sol = solve_ivp(
                system_func,
                [0, self.iterations * self.dt],
                initial_state,
                method='RK45',
                t_eval=np.linspace(0, self.iterations * self.dt, self.iterations)
            )
            features = self._extract_features(sol.y)
            results.append(features)

        return torch.tensor(results, dtype=torch.float32)

# Define EnhancedLCMNet class
class EnhancedLCMNet(nn.Module):
    def __init__(self):
        super(EnhancedLCMNet, self).__init__()
        chaotic_features = 9
        self.fc1 = nn.Linear(2, 32)
        self.bn1 = nn.BatchNorm1d(32)

        self.chaotic_lorenz = EnhancedChaoticLayer(system='lorenz')
        self.chaotic_rossler = EnhancedChaoticLayer(system='rossler')
        self.chaotic_chen = EnhancedChaoticLayer(system='chen')

        combined_features = 32 + (chaotic_features * 3)
        self.fc2 = nn.Linear(combined_features, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.fc3 = nn.Linear(64, 32)
        self.bn3 = nn.BatchNorm1d(32)
        self.fc4 = nn.Linear(32, 1)

        self.dropout = nn.Dropout(0.1)
        self.leaky_relu = nn.LeakyReLU()

    def forward(self, x):
        x_hidden = self.leaky_relu(self.bn1(self.fc1(x)))

        lorenz_features = self.chaotic_lorenz(x)
        rossler_features = self.chaotic_rossler(x)
        chen_features = self.chaotic_chen(x)

        combined = torch.cat([x_hidden, lorenz_features, rossler_features, chen_features], dim=1)

        x_out = self.leaky_relu(self.bn2(self.fc2(combined)))
        x_out = self.dropout(x_out)
        x_out = self.leaky_relu(self.bn3(self.fc3(x_out)))
        x_out = self.fc4(x_out)

        return x_out  # Remove torch.abs constraint

# Training function
def train_model(model, train_inputs, train_targets, val_inputs, val_targets, test_inputs, test_targets, epochs=1000):
    criterion = nn.HuberLoss()  # Updated loss function
    optimizer = RAdam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(train_inputs)
        loss = criterion(outputs, train_targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(val_inputs)
            val_loss = criterion(val_outputs, val_targets)

            # Include test set evaluation
            test_outputs = model(test_inputs)
            test_loss = criterion(test_outputs, test_targets)

        scheduler.step(val_loss)

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}, Test Loss: {test_loss.item():.4f}")

    return model

# Main script
if __name__ == "__main__":
    num_samples = 300
    inputs = torch.randint(1, 50, (num_samples, 2)).float()
    lcm_targets = torch.tensor([compute_lcm(a, b) for a, b in inputs.numpy()], dtype=torch.float32).unsqueeze(1)

    input_scaler = StandardScaler()
    inputs = torch.tensor(input_scaler.fit_transform(inputs), dtype=torch.float32)

    train_size = int(0.7 * num_samples)
    val_size = int(0.15 * num_samples)
    train_inputs, val_inputs, test_inputs = inputs[:train_size], inputs[train_size:train_size+val_size], inputs[train_size+val_size:]
    train_targets, val_targets, test_targets = lcm_targets[:train_size], lcm_targets[train_size:train_size+val_size], lcm_targets[train_size+val_size:]

    model = EnhancedLCMNet()
    model = train_model(model, train_inputs, train_targets, val_inputs, val_targets, test_inputs, test_targets)

    model.eval()
    with torch.no_grad():
        test_outputs = model(test_inputs)
        test_loss = nn.HuberLoss()(test_outputs, test_targets)  # Consistent loss function
        print(f"\nFinal Test Loss: {test_loss.item():.4f}")

        test_outputs_np = test_outputs.numpy()
        test_targets_np = test_targets.numpy()

        print("\nSample Predictions:")
        for i in range(10):
            orig_inputs = input_scaler.inverse_transform(test_inputs[i].numpy().reshape(1, -1))[0]
            print(f"Numbers: ({int(orig_inputs[0])}, {int(orig_inputs[1])}), Predicted LCM: {test_outputs_np[i][0]:.1f}, True LCM: {test_targets_np[i][0]}")

        mape = np.mean(np.abs((test_targets_np - test_outputs_np) / test_targets_np)) * 100
        print(f"\nMean Absolute Percentage Error: {mape:.2f}%")

Epoch 0, Train Loss: 458.5773, Val Loss: 437.5904, Test Loss: 402.1864
Epoch 50, Train Loss: 458.4429, Val Loss: 436.9198, Test Loss: 401.4741
Epoch 100, Train Loss: 458.3235, Val Loss: 436.7776, Test Loss: 401.3283
Epoch 150, Train Loss: 458.2747, Val Loss: 436.6756, Test Loss: 401.2248
Epoch 200, Train Loss: 458.2658, Val Loss: 436.6471, Test Loss: 401.1971
Epoch 250, Train Loss: 458.2648, Val Loss: 436.6393, Test Loss: 401.1894
Epoch 300, Train Loss: 458.2613, Val Loss: 436.6403, Test Loss: 401.1900
Epoch 350, Train Loss: 458.2670, Val Loss: 436.6392, Test Loss: 401.1900
Epoch 400, Train Loss: 458.2611, Val Loss: 436.6417, Test Loss: 401.1919
Epoch 450, Train Loss: 458.2595, Val Loss: 436.6421, Test Loss: 401.1924
Epoch 500, Train Loss: 458.2515, Val Loss: 436.6395, Test Loss: 401.1906
Epoch 550, Train Loss: 458.2569, Val Loss: 436.6418, Test Loss: 401.1922
Epoch 600, Train Loss: 458.2599, Val Loss: 436.6396, Test Loss: 401.1887
Epoch 650, Train Loss: 458.2585, Val Loss: 436.6376, T